# hex_color

Perceptual colour-classification: given a hue-jittered `#RRGGBB` code, name the colour it best matches from six fixed choices (red/orange/yellow/green/blue/purple). Singleton task; the answer variable `color` carries a periodic hue-centre embedding (360 deg period).

`indigo` was dropped from the source's seven classes (Qwen3-4B collapses it into purple); see README.md.

CPU-only notebook - it demonstrates the *task* (causal model, samples, token positions, counterfactuals), never a language model.

In [ ]:
from causalab.tasks.hex_color import CAUSAL_MODEL, COUNTERFACTUAL_GENERATORS
from causalab.tasks.hex_color.config import COLORS, HUE_CENTERS_DEG, HUE_PERIOD

## Causal Model Variables

In [ ]:
print("variables:", CAUSAL_MODEL.variables)
print("inputs:", CAUSAL_MODEL.inputs)
print("# stimuli (hex values):", len(CAUSAL_MODEL.values["hex"]))
print("colours:", CAUSAL_MODEL.values["color"])
print("period (deg):", CAUSAL_MODEL.periods, "| HUE_PERIOD =", HUE_PERIOD)
print("hue centres (deg):", HUE_CENTERS_DEG)
print(
    "embedding of each colour (hue centre):",
    {c: CAUSAL_MODEL.embeddings["color"](c) for c in COLORS},
)

## Sample Generation

In [ ]:
for _ in range(3):
    s = CAUSAL_MODEL.sample_input()
    print(repr(s["raw_input"]), "->", repr(s["raw_output"]))

## Token Positions

Two positions are materialised at experiment time via `token_positions.create_token_positions(pipeline)` (needs a tokenizer, so not instantiated in this model-free notebook):

- **`last_token`** - the final prompt token (where the answer is read off).
- **`hex`** - the last token of the `#RRGGBB` stimulus span.

In [ ]:
import inspect
from causalab.tasks.hex_color import token_positions

print(inspect.getsource(token_positions.create_token_positions))

## Counterfactual Generation

In [ ]:
import random

random.seed(0)
gen = COUNTERFACTUAL_GENERATORS["different_color"]
ex = gen()
base, cf = ex["input"], ex["counterfactual_inputs"][0]
print("base   :", repr(base["raw_input"]), "->", base["color"])
print("counter:", repr(cf["raw_input"]), "->", cf["color"])
print("generators:", list(COUNTERFACTUAL_GENERATORS))